In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px

# =========================
# PROJECT PATHS
# =========================

PROJECT_ROOT = Path.cwd().parents[1]
DATA_FEB_DIR = PROJECT_ROOT / "Dataset_feb"
SAMSUNG_DIR = DATA_FEB_DIR / "Dataset_samsung"

input_file = SAMSUNG_DIR / "Geo_file.csv"
output_excel = SAMSUNG_DIR / "pontos_altitude_alta.xlsx"
output_html = SAMSUNG_DIR / "mapa_pontos_altitude_alta.html"

print("Input file found:", input_file.exists())


# =========================
# PARAMETERS
# =========================

ALT_THRESHOLD = 160


# =========================
# LOAD FILE
# =========================

df = pd.read_csv(input_file)


# =========================
# COLUMN NAMES
# =========================

LAT_COL = "latitude"
LON_COL = "longitude"
ALT_COL = "altitude"


# =========================
# CONVERT TO NUMERIC
# =========================

df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
df[ALT_COL] = pd.to_numeric(df[ALT_COL], errors="coerce")

# Remove rows with missing altitude or coordinates
df = df.dropna(subset=[ALT_COL, LAT_COL, LON_COL]).copy()


# =========================
# FILTER POINTS ABOVE THRESHOLD
# =========================

high_alt_df = df[df[ALT_COL] > ALT_THRESHOLD].copy()


# =========================
# SAVE TO EXCEL
# =========================

high_alt_df.to_excel(output_excel, index=False)


# =========================
# SUMMARY
# =========================

print(f"Total points analysed: {len(df)}")
print(f"Points with altitude > {ALT_THRESHOLD} m: {len(high_alt_df)}")
print(f"Excel file created: {output_excel}")


# =========================
# CREATE MAP
# =========================

if len(high_alt_df) > 0:
    fig = px.scatter_mapbox(
        high_alt_df,
        lat=LAT_COL,
        lon=LON_COL,
        color=ALT_COL,
        hover_data=[ALT_COL],
        zoom=10,
        height=700,
        title=f"Points with altitude > {ALT_THRESHOLD} m"
    )

    fig.update_layout(mapbox_style="open-street-map")
    fig.write_html(output_html)

    print(f"Map created: {output_html}")

else:
    print("No points were found above the altitude threshold.")

Display the 10 highest altitude values

In [ ]:
df["altitude"].sort_values(ascending=False).head(10)

Display the maximum altitude among the filtered points

In [ ]:

max_altitude = high_alt_df["altitude"].max()
print(f"Maximum altitude: {max_altitude} m")

Generate altitude distribuiton histogram

In [ ]:
import matplotlib.pyplot as plt

# Load the geographic dataset
df = pd.read_csv(input_file)

# Convert altitude values to numeric
df["altitude"] = pd.to_numeric(df["altitude"], errors="coerce")

# Remove missing altitude values
df = df.dropna(subset=["altitude"])

# Plot altitude distribution
plt.figure(figsize=(10, 6))
plt.hist(df["altitude"], bins=50)
plt.xlabel("Altitude (m)")
plt.ylabel("Frequency")
plt.title("Altitude Distribution")
plt.grid(True)
plt.show()

Identification of extreme altitude values

In [ ]:
outliers = df[df["altitude"] > 1000].copy()

print(outliers[["latitude", "longitude", "altitude"]].head(20))
print(f"Total points above 1000 m: {len(outliers)}")

Map of extreme altitude outliers

In [ ]:
import plotly.express as px

outliers = df[df["altitude"] > 1000].copy()

if not outliers.empty:
    fig = px.scatter_mapbox(
        outliers,
        lat="latitude",
        lon="longitude",
        color="altitude",
        hover_data=["altitude"],
        height=700,
        title="GPS points with altitude above 1000 m"
    )

    fig.update_layout(
        mapbox_style="open-street-map",
        mapbox_center={
            "lat": outliers["latitude"].mean(),
            "lon": outliers["longitude"].mean()
        },
        mapbox_zoom=10
    )

    fig.show()
else:
    print("No points above 1000 m were found.")